# 第9章 質問応答

## 9.4 文書検索モデルの実装

### 9.4.3 BPRの実装

#### 準備

In [1]:
!pip install 'datasets<4.0.0' torch 'transformers[ja,torch]<4.41.0' 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 137.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 58.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 155.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 MB 37.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from transformers.trainer_utils import set_seed

# 乱数のシードを設定する
set_seed(42)

#### データセットの読み込みと前処理

In [4]:
from datasets import load_dataset

# Hugging Face Hubのllm-book/aio-retrieverのリポジトリから
# AI王データセットの訓練セットを読み込む
train_dataset = load_dataset("llm-book/aio-retriever", split="train")

Generating train split:   0%|          | 0/22335 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
# 読み込まれた訓練セットの形式と事例数を確認する
print(train_dataset)

Dataset({
    features: ['qid', 'competition', 'timestamp', 'section', 'number', 'original_question', 'original_answer', 'original_additional_info', 'question', 'answers', 'passages', 'positive_passage_indices', 'negative_passage_indices'],
    num_rows: 22335
})


In [6]:
from pprint import pprint

# 読み込まれた訓練セットの内容を確認する
pprint(train_dataset[0])

{'answers': ['26文字'],
 'competition': 'abc ～the first～',
 'negative_passage_indices': [1,
                              2,
                              3,
                              4,
                              5,
                              6,
                              7,
                              8,
                              9,
                              10,
                              11,
                              12,
                              13,
                              14,
                              15,
                              16,
                              17,
                              18,
                              19,
                              20,
                              21,
                              22,
                              23,
                              24,
                              25,
                              26,
                              27,
                              28,


In [9]:
print(train_dataset[0]["passages"])

[{'passage_id': 741177, 'title': 'ビヨンセ', 'text': 'アメリカのイラストレーターのVivian Lohが、ビヨンセのビデオクリップから様々なポーズを抽出し、アルファベット26文字をイラストレーションにしたフォント「The ABC’s of Beyoncé」を製作している。'}, {'passage_id': 2185313, 'title': 'ABC記譜法', 'text': 'ABC記譜法( - きふほう、ABC music notation あるいは ABC notation)は、パソコン等で使われる音楽記述言語の一つで、イギリスのChris Walshawによって考案された。単に「ABC」とも、また小文字で「abc」とも言う。音高を表すアルファベットと、音長を表す数字、その他の若干の記号を組み合わせて表記する。'}, {'passage_id': 569261, 'title': 'アメリカン・ブロードキャスティング・カンパニー', 'text': ' ABC Inc. (以前の Capital Cities/ABC Inc.)はABCの直接の親会社であり、オーナーはディズニーである。「ABC」はアルファベットの最初の3文字であることから、ネットワークはしばしば「アルファベット・ネットワーク Alphabet Network」とも呼ばれる。同じくABCを略称に用いる日本の朝日放送グループホールディングス(及び傘下の朝日放送テレビ・朝日放送ラジオ、前身の朝日放送の英称Asahi Broadcasting Corporationから)や、オーストラリアのオーストラリア放送協会(Australian Broadcasting Corporation)との関係は特にない。なお、区別のため本局を「米ABC放送」と表記することもある。日本ではフジテレビジョンと業務提携している。'}, {'passage_id': 282845, 'title': 'ABC', 'text': 'ABC(エービーシー)は、ラテン文字アルファベットの最初の3文字。物事を習得する際の初歩。アルファベットの最初の3文字であることから転じている。日本語における「いろは」の比喩的表現と同様である。'}, {'passage_id': 2375217, 